# M3L4 E06 — Evaluator Agent simple
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

---

## Qué necesitas saber antes

| Módulo | Concepto | Por qué lo necesitas acá |
|---|---|---|
| M3L4 E04-E05 | Golden datasets, evaluación de routing | Evaluar respuestas es el siguiente nivel: no solo "llegó al agente correcto" sino "respondió bien" |
| M3L1 | Tool contracts determinísticas | El evaluador por keywords es una tool contract: input -> output estable |
| M3L3 | LLM-as-judge | El esqueleto de evaluador con LLM que ves al final se conecta con M3L3 |
| Python | `str.lower()`, `any()`, list comprehension | Para matcheo case-insensitive de keywords |

---

## Definiciones clave

| Concepto | Definición simple | Cómo aparece en este notebook |
|---|---|---|
| **Evaluador determinístico** | Función que puntúa respuestas basándose en reglas fijas (keywords) | `simple_keyword_evaluator()` |
| **Score** | Valor numérico entre 0.0 y 1.0 que mide calidad | `score = matched / total_keywords` |
| **Matched keywords** | Palabras esperadas que aparecen en la respuesta del agente | `[kw for kw in expected if kw.lower() in actual.lower()]` |
| **LLM-as-judge** | Evaluador que usa un LLM para juzgar calidad semánticamente | `llm_evaluator_prompt()` genera el prompt para el LLM |
| **Quality score** | Promedio de scores sobre un conjunto de casos | `df['score'].mean()` |

---

## Cómo encaja esto en un sistema de agentes

```
E04-E05: El router envía al agente correcto
    |  "llegó al destino adecuado"
    v
E06: El agente responde bien (ESTE EJERCICIO)
    |  "la respuesta es correcta, completa y clara"
    v
Métricas de calidad:
    |  Score determinístico (keywords)
    |  Score LLM-as-judge (semántico)
    v
E11-E12: Dashboards de calidad y mejora continua
```

**Objetivo del ejercicio:** crear un evaluador automático que puntúa la calidad de las respuestas de los agentes.

### Por qué evaluar respuestas?

Medir el routing accuracy no es suficiente. También necesitamos saber:

- La respuesta contiene la información correcta?
- Es clara y completa?
- No alucina datos que no existen?

### Dos enfoques

| Evaluador | Ventaja | Limitación |
|---|---|---|
| **Determinístico** (keywords) | Rápido, reproducible, sin LLM | Limitado a vocabulario fijo |
| **LLM-as-judge** | Entiende semántica, contexto | Costo, latencia, necesita calibración |

## Instalación e imports

| Import | Qué hace | Por qué lo necesitamos |
|---|---|---|
| `pandas` (via pip) | DataFrames para mostrar resultados de evaluación | Para tabla de scores por caso |
| Ningún LLM | El evaluador determinístico no necesita API externa | Es rápido, gratuito y reproducible |

```python
!pip install pandas -q
import pandas as pd
```

## Parte 1 — Evaluador determinístico por keywords

Mide qué porcentaje de las palabras clave esperadas aparecen en la respuesta del agente.

### Algoritmo

```python
# Entrada:
expected_keywords = ['factura', 'pagos', 'portal']
actual_answer = 'Podés ver tu factura desde el portal de pagos.'

# Matching (case insensitive):
# 'factura'  -> sí, aparece
# 'pagos'    -> sí, aparece
# 'portal'   -> sí, aparece
# matched = 3 de 3

# Score:
score = 3 / 3 = 1.0
```

### Desglose de `simple_keyword_evaluator(expected_keywords, actual_answer)`

| Parámetro | Tipo | Qué es |
|---|---|---|
| `expected_keywords` | `list[str]` | Palabras que DEBERÍAN estar en una respuesta correcta |
| `actual_answer` | `str` | La respuesta que generó el agente |
| **Retorna** | `dict` | `{'score': 0.67, 'matched_keywords': ['factura', 'portal'], 'reason': '...'}` |

In [ ]:
def simple_keyword_evaluator(expected_keywords: list, actual_answer: str) -> dict:
    """
    Evalúa si la respuesta del agente contiene las keywords esperadas.

    Args:
        expected_keywords: lista de palabras que deben estar en la respuesta
        actual_answer: respuesta generada por el agente

    Returns:
        dict con score (0.0-1.0), matched_keywords y reason
    """
    # TODO 1: manejar el caso donde expected_keywords está vacía
    # devolver score 0, reason 'No expected keywords provided.'

    # TODO 2: verificar cuáles keywords están en actual_answer (case insensitive)
    matched = []

    # TODO 3: calcular score = matched / total keywords (redondeado a 2 decimales)
    score = 0

    # TODO 4: retornar dict con score, matched_keywords y reason
    return {}

print('Función definida.')

## Casos de prueba

Probamos el evaluador con 4 escenarios:

| Caso | Keywords esperadas | Respuesta del agente | Score esperado |
|---|---|---|---|
| 1 (perfecto) | factura, pagos, portal | "Podés ver tu factura desde el portal de pagos." | 1.0 |
| 2 (parcial) | vacaciones, solicitud, formulario, portal | "Para pedir vacaciones completa el formulario." | 0.5 (2/4) |
| 3 (incorrecto) | factura, pagos, portal | "Probá reiniciar la app." | 0.0 |
| 4 (sin keywords) | [] | cualquier cosa | 0.0 |

In [ ]:
# Caso 1: respuesta perfecta
result1 = simple_keyword_evaluator(
    expected_keywords=['factura', 'pagos', 'portal'],
    actual_answer='Podés ver tu factura desde el portal de pagos.'
)
print('Caso 1 (perfecto):', result1)

In [ ]:
# Caso 2: respuesta parcial
result2 = simple_keyword_evaluator(
    expected_keywords=['vacaciones', 'solicitud', 'formulario', 'portal'],
    actual_answer='Para pedir vacaciones completa el formulario.'
)
print('Caso 2 (parcial):', result2)

In [ ]:
# Caso 3: respuesta incorrecta
result3 = simple_keyword_evaluator(
    expected_keywords=['factura', 'pagos', 'portal'],
    actual_answer='Probá reiniciar la app.'
)
print('Caso 3 (incorrecto):', result3)

In [ ]:
# Caso 4: sin keywords esperadas
result4 = simple_keyword_evaluator(
    expected_keywords=[],
    actual_answer='Probá reiniciar la app.'
)
print('Caso 4 (sin keywords):', result4)

## Parte 2 — Evaluar el golden dataset completo

Aplicamos el evaluador a 5 casos reales con respuestas de agentes (algunas correctas, otras no).

In [ ]:
import pandas as pd

# Respuestas simuladas de los agentes (algunas correctas, algunas no)
evaluation_cases = [
    {
        'query': 'Necesito ver mi factura del mes pasado',
        'expected_keywords': ['factura', 'portal', 'pagos'],
        'agent_response': 'Podés ver tu factura desde el portal de pagos.'
    },
    {
        'query': 'Cómo solicito mis días de vacaciones?',
        'expected_keywords': ['vacaciones', 'portal', 'formulario'],
        'agent_response': 'Para solicitar vacaciones ingresa al portal de RRHH y completa el formulario.'
    },
    {
        'query': 'Mi VPN no conecta desde ayer',
        'expected_keywords': ['VPN', 'conexión', 'soporte'],
        'agent_response': 'Probá reiniciar el router.'
    },
    {
        'query': 'Cuándo se procesa el reembolso de gastos?',
        'expected_keywords': ['reembolso', 'gastos', '48 horas'],
        'agent_response': 'Los gastos se procesan dentro de las 48 horas hábiles.'
    },
    {
        'query': 'Necesito el contrato de confidencialidad',
        'expected_keywords': ['contrato', 'confidencialidad', 'legal'],
        'agent_response': 'Completa el formulario de RRHH para acceder al contrato.'
    },
]

print(f'Casos de evaluación: {len(evaluation_cases)}')

In [ ]:
# TODO: evaluar cada caso y mostrar tabla de resultados
results = []
for case in evaluation_cases:
    eval_result = simple_keyword_evaluator(
        case['expected_keywords'],
        case['agent_response']
    )
    results.append({
        'query': case['query'][:40],
        'score': eval_result.get('score'),
        'matched': eval_result.get('matched_keywords'),
        'reason': eval_result.get('reason')
    })

df = pd.DataFrame(results)
print(f'Quality score promedio: {df["score"].mean():.2f}')
df

## Parte 3 — Esqueleto del evaluador LLM-as-judge (extensión)

Este código es **conceptual** — requiere un LLM conectado. Lo verás en producción en ejercicios avanzados.

### Diferencias con el evaluador determinístico

| Aspecto | Keywords | LLM-as-judge |
|---|---|---|
| Qué mide | Presencia de palabras | Corrección, completitud, claridad |
| Idioma | Agnóstico (case insensitive) | Entiende semántica |
| Falsos positivos | Keyword aparece pero en contexto incorrecto | Detecta contexto |
| Reproducibilidad | 100% | Depende del LLM y temperatura |
| Costo | 0 | Por llamada al LLM |

In [ ]:
def llm_evaluator_prompt(query: str, actual_answer: str) -> str:
    """Genera el prompt para un evaluador LLM."""
    return f"""Evalúa la siguiente respuesta de un agente de soporte interno.

Query del usuario:
{query}

Respuesta del agente:
{actual_answer}

Criterios de evaluación:
- Corrección factual (la información es correcta?)
- Completitud (responde la pregunta completa?)
- Claridad (es fácil de entender?)
- No alucinación (no inventa datos?)

Responde SOLO con JSON sin markdown:
{{"score": 1-10, "reason": "explicación breve en una oración"}}
"""

# Mostrar el prompt para el primer caso
print(llm_evaluator_prompt(
    'Cómo solicito mis días de vacaciones?',
    'Para solicitar vacaciones ingresa al portal de RRHH y completa el formulario.'
))

In [ ]:
r = simple_keyword_evaluator(['factura', 'pagos', 'portal'], 'Podés ver tu factura desde el portal de pagos.')
assert r['score'] == 1.0, f'Score esperado 1.0, obtenido {r["score"]}'
assert len(r['matched_keywords']) == 3

r2 = simple_keyword_evaluator([], 'cualquier cosa')
assert r2['score'] == 0

print('Checks E06 OK')

## Errores comunes

| Error | Causa | Cómo detectarlo |
|---|---|---|
| Score siempre 0 | No implementar el matching de keywords | `matched` siempre vacío |
| Score siempre 1.0 | No hacer case-insensitive | Keywords en mayúsculas no matchean |
| Division by zero | `expected_keywords` vacío sin manejo especial | Agregar `if not expected_keywords: return {'score': 0, ...}` |
| Matcheo parcial como fallo total | Esperar que TODAS las keywords aparezcan | El score refleja proporción, no condición binaria |
| Reason poco descriptiva | Devolver solo el score sin explicación | El evaluador debe decir qué keywords matchearon y cuáles no |

## Síntesis

### Qué construiste

| Componente | Descripción |
|---|---|
| `simple_keyword_evaluator()` | Función que puntúa respuestas basándose en palabras clave esperadas |
| Evaluación por casos | Prueba con 4 escenarios (perfecto, parcial, incorrecto, sin keywords) |
| Quality score promedio | Métrica global sobre el golden dataset de respuestas |
| Esqueleto LLM-as-judge | Prompt para evaluación semántica con un LLM |

### Dos niveles de evaluación

```
Nivel 1: Routing accuracy (E04-E05)
    "El query llegó al agente correcto?"
    Métrica: proportion de casos con intent correcto

Nivel 2: Quality score (E06)
    "La respuesta del agente es buena?"
    Métrica: score de keywords o evaluación LLM
```

### Relación con otros ejercicios

| Ejercicio | Conexión con E06 |
|---|---|
| **E07** | Dashboard de métricas: visualizar quality scores en el tiempo |
| **E11** | Golden dataset scores: integrar evaluación de respuestas al benchmark global |
| **E12** | Ciclo de mejora: medir calidad, identificar agentes débiles, mejorar prompts |